A következő jegyzetfüzetet a GitHub Copilot Chat automatikusan generálta, és csak a kezdeti beállításhoz készült


# Bevezetés a Prompt Mérnökségbe
A prompt mérnökség a természetes nyelvfeldolgozási feladatokhoz megfelelő promptok tervezésének és optimalizálásának folyamata. Ez magában foglalja a megfelelő promptok kiválasztását, azok paramétereinek hangolását, valamint a teljesítményük értékelését. A prompt mérnökség elengedhetetlen a magas pontosság és hatékonyság eléréséhez az NLP modellekben. Ebben a szakaszban az OpenAI modellek használatával ismerkedünk meg a prompt mérnökség alapjaival.


### 1. Gyakorlat: Tokenizáció
Ismerkedjen meg a Tokenizációval a tiktoken segítségével, egy nyílt forráskódú, gyors tokenizáló az OpenAI-tól
További példákért tekintse meg az [OpenAI Cookbook](https://github.com/openai/openai-cookbook/blob/main/examples/How_to_count_tokens_with_tiktoken.ipynb?WT.mc_id=academic-105485-koreyst) oldalt.


In [ ]:
# EXERCISE:
# 1. Run the exercise as is first
# 2. Change the text to any prompt input you want to use & re-run to see tokens

import tiktoken

# Define the prompt you want tokenized
text = f"""
Jupiter is the fifth planet from the Sun and the \
largest in the Solar System. It is a gas giant with \
a mass one-thousandth that of the Sun, but two-and-a-half \
times that of all the other planets in the Solar System combined. \
Jupiter is one of the brightest objects visible to the naked eye \
in the night sky, and has been known to ancient civilizations since \
before recorded history. It is named after the Roman god Jupiter.[19] \
When viewed from Earth, Jupiter can be bright enough for its reflected \
light to cast visible shadows,[20] and is on average the third-brightest \
natural object in the night sky after the Moon and Venus.
"""

# Set the model you want encoding for
encoding = tiktoken.encoding_for_model("gpt-4o")

# Encode the text - gives you the tokens in integer form
tokens = encoding.encode(text)
print(tokens);

# Decode the integers to see what the text versions look like
[encoding.decode_single_token_bytes(token) for token in tokens]


### 2. gyakorlat: OpenAI API kulcs beállításának ellenőrzése

Futtassa az alábbi kódot annak ellenőrzésére, hogy az OpenAI végpontja helyesen van-e beállítva. A kód egy egyszerű alap promptot próbál ki, és validálja a válasz befejezését. A „oh say can you see” bemenetnek a „by the dawn's early light..” körüli befejezéssel kell folytatódnia.


In [ ]:
# Uses the OpenAI client against the Azure OpenAI (Microsoft Foundry) v1 endpoint
# with the Responses API. See https://aka.ms/openai/start

import os
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()

client = OpenAI(
  api_key=os.environ['AZURE_OPENAI_API_KEY'],
  base_url=f"{os.environ['AZURE_OPENAI_ENDPOINT'].rstrip('/')}/openai/v1/",
  )

deployment=os.environ['AZURE_OPENAI_DEPLOYMENT']

def get_completion(prompt):
    response = client.responses.create(
        model=deployment,
        input=prompt,
        temperature=0, # this is the degree of randomness of the model's output
        max_output_tokens=1024,
        store=False,
    )
    return response.output_text

## ---------- Call the helper method

### 1. Set primary content or prompt text
text = f"""
oh say can you see
"""

### 2. Use that in the prompt template below
prompt = f"""
```{text}```
"""

## 3. Run the prompt
response = get_completion(prompt)
print(response)


### 3. gyakorlat: Hamis információk
Vizsgáld meg, mi történik, ha arra kéred az LLM-et, hogy egy olyan témáról adjon választ, amely lehet, hogy nem létezik, vagy olyan témákról, amelyekről nem tudhat, mert kívül esnek a korábban betanított adatbázisán (frissebb témák). Nézd meg, hogyan változik a válasz, ha másik promptot vagy másik modellt próbálsz ki.


In [ ]:

## Set the text for simple prompt or primary content
## Prompt shows a template format with text in it - add cues, commands etc if needed
## Run the completion 
text = f"""
generate a lesson plan on the Martian War of 2076.
"""

prompt = f"""
```{text}```
"""

response = get_completion(prompt)
print(response)

### 4. gyakorlat: Utasítás alapú 
Használd a "text" változót az elsődleges tartalom beállításához, 
és a "prompt" változót az elsődleges tartalommal kapcsolatos utasítás megadásához.

Itt arra kérjük a modellt, hogy foglalja össze a szöveget egy másodikos diák számára.


In [ ]:
# Test Example
# https://platform.openai.com/playground/p/default-summarize

## Example text
text = f"""
Jupiter is the fifth planet from the Sun and the \
largest in the Solar System. It is a gas giant with \
a mass one-thousandth that of the Sun, but two-and-a-half \
times that of all the other planets in the Solar System combined. \
Jupiter is one of the brightest objects visible to the naked eye \
in the night sky, and has been known to ancient civilizations since \
before recorded history. It is named after the Roman god Jupiter.[19] \
When viewed from Earth, Jupiter can be bright enough for its reflected \
light to cast visible shadows,[20] and is on average the third-brightest \
natural object in the night sky after the Moon and Venus.
"""

## Set the prompt
prompt = f"""
Summarize content you are provided with for a second-grade student.
```{text}```
"""

## Run the prompt
response = get_completion(prompt)
print(response)

### 5. gyakorlat: Összetett utasítás  
Próbálj ki egy olyan kérést, amely tartalmaz rendszer-, felhasználói és asszisztensi üzeneteket  
A rendszer beállítja az asszisztens kontextusát  
A felhasználói és asszisztensi üzenetek többkörös beszélgetési kontextust adnak  

Figyeld meg, hogy az asszisztens személyisége "szarkasztikus"-ra van állítva a rendszer kontextusában.  
Próbálj ki egy másik személyiségkontextust. Vagy próbálj ki egy másik bemenet/kimenet üzenetsorozatot  


In [ ]:
response = client.responses.create(
    model=deployment,
    input=[
        {"role": "system", "content": "You are a sarcastic assistant."},
        {"role": "user", "content": "Who won the world series in 2020?"},
        {"role": "assistant", "content": "Who do you think won? The Los Angeles Dodgers of course."},
        {"role": "user", "content": "Where was it played?"}
    ],
    store=False,
)
print(response.output_text)


### Gyakorlat: Fedezd fel az intuíciódat
A fenti példák mintákat adnak, amelyeket új promptok létrehozásához használhatsz (egyszerű, összetett, utasításos stb.) – próbálj meg más gyakorlatokat alkotni, hogy felfedezz néhány további ötletet, amikről beszéltünk, például példákat, jelzéseket és egyebeket.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Jogi nyilatkozat**:
Ez a dokumentum az AI fordítási szolgáltatás, a [Co-op Translator](https://github.com/Azure/co-op-translator) segítségével készült. Bár az pontosságra törekszünk, kérjük, vegye figyelembe, hogy az automatikus fordítások hibákat vagy pontatlanságokat tartalmazhatnak. Az eredeti dokumentum az anyanyelvén tekintendő hiteles forrásnak. Fontos információk esetén professzionális emberi fordítást javasolunk. Nem vállalunk felelősséget semmilyen félreértésért vagy téves értelmezésért, amely ebből a fordításból ered.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
